# Swin-B — From Scratch in PyTorch
**Paper:** Swin Transformer: Hierarchical Vision Transformer using Shifted Windows (Liu et al., ICCV 2021)

| Config | Value |
|--------|-------|
| Version | Swin V1 |
| Image Size | 224x224 |
| embed_dim | 128 |
| depths | [2, 2, 18, 2] |
| num_heads | [4, 8, 16, 32] |
| window_size | 7 |
| Parameters | ~88M |
| Top-1 (ImageNet) | 83.5% |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import math
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

In [ ]:
# Cell 2 — Swin-B Architecture (from scratch)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)


def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)


class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=4, in_channels=3, embed_dim=96):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return self.norm(x), H, W


class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm      = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x, H, W):
        B, _, C = x.shape
        x  = x.view(B, H, W, C)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x  = torch.cat([x0, x1, x2, x3], dim=-1).view(B, -1, 4 * C)
        return self.reduction(self.norm(x)), H // 2, W // 2


class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads   = num_heads
        self.window_size = window_size
        self.scale       = (dim // num_heads) ** -0.5

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2*window_size-1) * (2*window_size-1), num_heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords   = torch.stack(torch.meshgrid([coords_h, coords_w], indexing='ij'))
        coords_f = torch.flatten(coords, 1)
        rel      = coords_f[:, :, None] - coords_f[:, None, :]
        rel      = rel.permute(1, 2, 0).contiguous()
        rel[:, :, 0] += window_size - 1
        rel[:, :, 1] += window_size - 1
        rel[:, :, 0] *= 2 * window_size - 1
        self.register_buffer('relative_position_index', rel.sum(-1))

        self.qkv       = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj      = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale

        rpb  = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(self.window_size**2, self.window_size**2, -1)
        attn = attn + rpb.permute(2, 0, 1).unsqueeze(0)

        if mask is not None:
            nW   = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.attn_drop(F.softmax(attn, dim=-1))
        x    = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        return self.proj_drop(self.proj(x))


class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., dropout=0., attn_drop=0.):
        super().__init__()
        self.shift_size  = shift_size
        self.window_size = window_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = WindowAttention(dim, window_size, num_heads,
                                     attn_drop=attn_drop, proj_drop=dropout)
        self.norm2 = nn.LayerNorm(dim)
        mlp_dim    = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim), nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask):
        B, H, W, C = x.shape
        shortcut    = x
        x           = self.norm1(x)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))

        x_win = window_partition(x, self.window_size).view(-1, self.window_size**2, C)
        x_win = self.attn(x_win, mask=attn_mask)
        x     = window_reverse(
            x_win.view(-1, self.window_size, self.window_size, C), self.window_size, H, W)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x


class SwinStage(nn.Module):
    def __init__(self, dim, depth, num_heads, window_size,
                 mlp_ratio=4., dropout=0., attn_drop=0., downsample=None):
        super().__init__()
        self.window_size = window_size
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim, num_heads=num_heads, window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,
                mlp_ratio=mlp_ratio, dropout=dropout, attn_drop=attn_drop,
            ) for i in range(depth)
        ])
        self.downsample = downsample(dim) if downsample else None

    def forward(self, x, H, W):
        shift_size = self.window_size // 2 if min(H, W) > self.window_size else 0
        if shift_size > 0:
            img_mask = torch.zeros((1, H, W, 1), device=x.device)
            for h_s, w_s, cnt in [
                (slice(0, -self.window_size), slice(0, -self.window_size), 0),
                (slice(0, -self.window_size), slice(-self.window_size, -shift_size), 1),
                (slice(0, -self.window_size), slice(-shift_size, None), 2),
                (slice(-self.window_size, -shift_size), slice(0, -self.window_size), 3),
                (slice(-self.window_size, -shift_size), slice(-self.window_size, -shift_size), 4),
                (slice(-self.window_size, -shift_size), slice(-shift_size, None), 5),
                (slice(-shift_size, None), slice(0, -self.window_size), 6),
                (slice(-shift_size, None), slice(-self.window_size, -shift_size), 7),
                (slice(-shift_size, None), slice(-shift_size, None), 8),
            ]:
                img_mask[:, h_s, w_s, :] = cnt
            mw = window_partition(img_mask, self.window_size).view(-1, self.window_size**2)
            attn_mask = (mw.unsqueeze(1) - mw.unsqueeze(2))
            attn_mask = attn_mask.masked_fill(attn_mask != 0, -100.0).masked_fill(attn_mask == 0, 0.0)
        else:
            attn_mask = None

        B, _, C = x.shape
        x = x.view(B, H, W, C)
        for blk in self.blocks:
            x = blk(x, attn_mask if blk.shift_size > 0 else None)
        x = x.view(B, H * W, C)
        if self.downsample:
            x, H, W = self.downsample(x, H, W)
        return x, H, W


class SwinTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=4, in_channels=3, num_classes=1000,
                 embed_dim=96, depths=(2,2,6,2), num_heads=(3,6,12,24),
                 window_size=7, mlp_ratio=4., dropout=0., attn_drop=0.):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_drop    = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            SwinStage(
                dim=embed_dim * (2**i), depth=depths[i], num_heads=num_heads[i],
                window_size=window_size, mlp_ratio=mlp_ratio,
                dropout=dropout, attn_drop=attn_drop,
                downsample=PatchMerging if i < len(depths) - 1 else None,
            ) for i in range(len(depths))
        ])
        num_features = embed_dim * (2 ** (len(depths) - 1))
        self.norm = nn.LayerNorm(num_features)
        self.head = nn.Linear(num_features, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.bias, 0)
                nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        x, H, W = self.patch_embed(x)
        x = self.pos_drop(x)
        for layer in self.layers:
            x, H, W = layer(x, H, W)
        x = self.norm(x).mean(dim=1)
        return self.head(x)


def swin_b(num_classes=1000):
    return SwinTransformer(
        img_size=224, embed_dim=128,
        depths=(2, 2, 18, 2), num_heads=(4, 8, 16, 32),
        window_size=7, num_classes=num_classes,
    )


In [ ]:
NUM_CLASSES = 10
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

model = swin_b(num_classes=NUM_CLASSES).to(DEVICE)
print(model)

In [ ]:
dummy_input = torch.randn(2, 3, 224, 224).to(DEVICE)
output      = model(dummy_input)
print(f'Input  shape : {dummy_input.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{'='*40}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"{'='*40}")

In [ ]:
print(f"{'Layer':<50} {'Shape':<30} {'Params':>10}")
print('-' * 92)
total = 0
for name, param in model.named_parameters():
    n = param.numel()
    total += n
    print(f"{name:<50} {str(list(param.shape)):<30} {n:>10,}")
print('-' * 92)
print(f"{'TOTAL':<81} {total:>10,}")

---
## Training

In [ ]:
DATA_DIR   = './data'
BATCH_SIZE = 8

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/val',   transform=val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

CLASS_NAMES = train_dataset.classes
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {len(train_dataset)}')
print(f'Val size   : {len(val_dataset)}')

In [ ]:
EPOCHS = 20
LR     = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += outputs.max(1)[1].eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, 100.0 * correct / total

print(f"{'Epoch':<8} {'Tr Loss':<10} {'Tr Acc':<10} {'Val Loss':<10} {'Val Acc':<10}")
print('-' * 50)
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);  history['val_acc'].append(vl_acc)
    print(f'{epoch:<8} {tr_loss:<10.4f} {tr_acc:<10.2f} {vl_loss:<10.4f} {vl_acc:<10.2f}')

torch.save(model.state_dict(), 'swin_b_best.pth')
print('Model saved: swin_b_best.pth')

---
## Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Swin-B — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Inference

In [ ]:
from PIL import Image

def predict_image(image_path, top_k=5):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img_pil = Image.open(image_path).convert('RGB')
    tensor  = transform(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)
    top_probs, top_idx = torch.topk(probs, top_k, dim=1)
    top_probs = top_probs[0].cpu().tolist()
    top_idx   = top_idx[0].cpu().tolist()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img_pil); axes[0].axis('off'); axes[0].set_title('Input')
    labels = [CLASS_NAMES[i] for i in top_idx]
    axes[1].barh(labels[::-1], [p*100 for p in top_probs[::-1]])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    plt.tight_layout(); plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({top_probs[0]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        probs = F.softmax(model(images.to(DEVICE)), dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
all_probs  = np.concatenate(all_probs,  axis=0)
all_labels = np.concatenate(all_labels, axis=0)
print(f'Predictions shape : {all_probs.shape}')

In [ ]:
y_bin   = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
roc_auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    roc_auc_scores[CLASS_NAMES[i]] = roc_auc
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {roc_auc:.3f})')
macro_auc = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'Swin-B — ROC AUC (Macro AUC = {macro_auc:.3f})', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Macro AUC : {macro_auc:.4f}')